In [63]:
# Print the raw KEGG disease flat file so we can see the exact format
# of the PATHWAY section before trying to parse it
raw = kegg_get("get/H00409")
print(raw)


ENTRY       H00409                      Disease
NAME        Type 2 diabetes mellitus
DESCRIPTION Type 2 diabetes mellitus (T2DM) is characterized by chronic hyperglycemia due to insulin resistance of peripheral tissues (skeletal muscle, liver, adipose tissue) and insufficient compensatory insulin secretion by pancreatic beta cells. Both insulin resistance and beta cell dysfunction are thought to result from the complex interplay of many different pathways under the combined control of environmental and genetic factors. It is accepted that T2DM results from population aging and adverse environmental factors of the modern world which favor the development of obesity.
CATEGORY    Endocrine and metabolic disease
BRITE       Human diseases in ICD-11 classification [BR:br08403]
             05 Endocrine, nutritional or metabolic diseases
              Endocrine diseases
               Diabetes mellitus
                5A11  Type 2 diabetes mellitus
                 H00409  Type 2 diabetes me

In [1]:
# KEGG DATA EXTRACTION SCRIPT
# Builds a T2DM–Melanoma Knowledge Graph dataset from the KEGG REST API
# Outputs: T2DM_Melanoma_KG_Dataset_auto.xlsx (Sheet1 = nodes, Sheet2 = edges)


import requests    # used to make HTTP requests to the KEGG REST API
import pandas as pd  # used to organise data into tables and export to Excel
import time        # used to add a delay between API calls so we don't overload KEGG
import re          # imported for potential pattern matching (available if needed for parsing)

# Base URL for all KEGG REST API requests — every call starts with this
BASE = "https://rest.kegg.jp"

In [34]:
# HELPER FUNCTION: kegg_get
# Makes a single HTTP request to the KEGG REST API and returns the raw text
# response. Every other function calls this to fetch data.

def kegg_get(endpoint):
    url = f"{BASE}/{endpoint}"  # build the full URL by combining the base with the specific endpoint
    response = requests.get(url)  # send the HTTP GET request to KEGG and store the response
    response.raise_for_status()   # if KEGG returned an error code (e.g. 404, 500), stop and raise an exception
    time.sleep(0.3)               # wait 0.3 seconds before returning — KEGG asks for respectful rate limiting
    return response.text          # return the raw text content of the response (KEGG returns plain text, not JSON)



# HELPER FUNCTION: parse_link
# Parses KEGG "link" responses, which return tab-separated pairs like:
#   hsa04930    hsa:1234
# Used to get relationships between databases (e.g. genes linked to a pathway).

def parse_link(text):
    pairs = []  # initialise empty list to hold the parsed (source, target) pairs
    for line in text.strip().split("\n"):  # split the response into individual lines, removing leading/trailing whitespace
        if "\t" in line:                   # only process lines that contain a tab (blank lines won't have one)
            parts = line.split("\t")       # split the line at the tab character into two parts
            pairs.append((parts[0].strip(), parts[1].strip()))  # add the pair to the list, stripping extra whitespace
    return pairs  # return the complete list of pairs


# HELPER FUNCTION: get_gene_symbol_from_hsa_id
# Some genes in KEGG disease entries do not have a symbol on the line itself —
# instead they only show a numeric HSA ID like [HSA:169026].
# This function takes that numeric ID, fetches the full KEGG gene entry, and
# returns the official gene symbol (e.g. "SLC30A8").
# It is only called when the normal symbol extraction fails.

def get_gene_symbol_from_hsa_id(hsa_number):
    text = kegg_get(f"get/hsa:{hsa_number}")  # fetch the full KEGG human gene entry using the numeric HSA ID (e.g. hsa:169026)
    for line in text.split("\n"):             # loop through each line of the gene entry flat file
        if line.startswith("SYMBOL"):         # look for the SYMBOL line which contains the official gene symbol
            return line.replace("SYMBOL", "").strip().split(",")[0].strip()  # remove the "SYMBOL" label, strip whitespace, take the first symbol if multiple are listed (comma-separated), and strip again
    return f"HSA:{hsa_number}"  # if no SYMBOL line is found (should not happen), fall back to using the HSA ID as the name



# HELPER FUNCTION: extract_gene_symbol
# Handles the two different formats that gene tokens appear in KEGG disease
# flat files:
# Returns:
#   ("gene",    "BRAF")       — a real gene symbol
#   ("pathway", "hsa04141")   — a pathway cross-reference line
#   ("hsa_id",  "169026")     — gene with no symbol, needs API lookup
#   (None,       None)        — line should be skipped entirely

def extract_gene_symbol(content):
    parts = content.split()  # split the line into whitespace-separated tokens

    if len(parts) < 2:       # fewer than two tokens means nothing useful to extract
        return None, None

    first_token  = parts[0]  # first token is a numeric KEGG entry ID
    second_token = parts[1]  # second token is symbol, [HSA:...] tag, or a plain English word

    # Detect pathway cross-reference lines — these appear inside the GENE section
    # but point to related pathways rather than encoding a gene.
    # They look like: "04141  Protein processing in ER [PATH:hsa04141]"
    # The [PATH:hsaXXXXX] tag is the reliable indicator.
    path_match = re.search(r'\[PATH:(hsa\d+)\]', content)  # search anywhere on the line for a [PATH:hsa04141] tag
    if path_match:                                          # if found, this line is a pathway reference
        return "pathway", path_match.group(1)              # return the pathway ID (e.g. "hsa04141")

    # Detect Format 2 gene lines — no symbol, only [HSA:XXXXX]
    if second_token.startswith("[HSA:"):                   # second token is an HSA ID tag, not a symbol
        hsa_match = re.search(r'\[HSA:(\d+)\]', second_token)  # extract the numeric ID from inside the brackets
        if hsa_match:
            return "hsa_id", hsa_match.group(1)            # return the raw HSA number for API lookup
        return None, None                                  # malformed tag — skip

    # Format 1 — second token should be the gene symbol
    # Validate it looks like a real gene symbol: starts uppercase, contains only
    # uppercase letters, digits, and hyphens (e.g. BRAF, PIK3CA, P3R3URF-PIK3R3)
    if re.match(r'^[A-Z][A-Z0-9\-]+$', second_token):
        return "gene", second_token                        # valid gene symbol — return it

    return None, None                                      # anything else is unexpected — skip



# UPDATED: parse_disease_entry
# Collects a fourth category — related_pathways — from the pathway
# cross-reference lines embedded in the GENE section of KEGG disease entries.

def parse_disease_entry(text):
    result = {
        "name":             "",   # disease name
        "genes":            [],   # gene symbols associated with this disease
        "drugs":            [],   # KEGG drug IDs listed for this disease
        "pathways":         [],   # pathways listed in the PATHWAY section
        "related_pathways": [],   # pathways found as cross-references in the GENE section
    }
    current_section = None  # tracks which section we are currently reading

    for line in text.split("\n"):  # loop through each line of the flat-file response

        # ── NAME ──────────────────────────────────────────────────────────────
        if line.startswith("NAME"):
            result["name"] = line.replace("NAME", "").strip()  # extract disease name
            current_section = "NAME"

        # ── GENE ──────────────────────────────────────────────────────────────
        elif line.startswith("GENE"):
            current_section = "GENE"
            content = line.replace("GENE", "").strip()  # remove section label
            if content:
                kind, value = extract_gene_symbol(content)  # classify the line content
                if kind == "gene":                           # plain gene symbol
                    result["genes"].append(value)
                elif kind == "hsa_id":                       # numeric ID only — look up symbol
                    symbol = get_gene_symbol_from_hsa_id(value)
                    if symbol:
                        result["genes"].append(symbol)
                elif kind == "pathway":                      # pathway cross-reference
                    if value not in result["related_pathways"]:  # avoid duplicates
                        result["related_pathways"].append(value)

        # ── DRUG ──────────────────────────────────────────────────────────────
        elif line.startswith("DRUG"):
            current_section = "DRUG"
            content = line.replace("DRUG", "").strip()
            dr_match = re.search(r'\[DR:(D\d+)\]', content)  # look for [DR:D10025] tag
            if dr_match:
                result["drugs"].append(dr_match.group(1))    # store the D-number

        # ── PATHWAY ───────────────────────────────────────────────────────────
        elif line.startswith("PATHWAY"):
            current_section = "PATHWAY"
            content = line.replace("PATHWAY", "").strip()
            if content:
                pathway_id = content.split()[0]              # first token is the pathway ID
                result["pathways"].append(pathway_id)

        # ── Continuation lines (indented) ─────────────────────────────────────
        elif line.startswith(" ") and current_section:
            content = line.strip()
            if not content:                                  # skip blank lines
                continue

            if current_section == "GENE":
                kind, value = extract_gene_symbol(content)  # classify the continuation line
                if kind == "gene":
                    result["genes"].append(value)
                elif kind == "hsa_id":
                    symbol = get_gene_symbol_from_hsa_id(value)
                    if symbol:
                        result["genes"].append(symbol)
                elif kind == "pathway":                      # pathway cross-reference on a continuation line
                    if value not in result["related_pathways"]:
                        result["related_pathways"].append(value)

            elif current_section == "DRUG":
                dr_match = re.search(r'\[DR:(D\d+)\]', content)
                if dr_match:
                    result["drugs"].append(dr_match.group(1))

            elif current_section == "PATHWAY":
                pathway_id = content.split()[0]
                result["pathways"].append(pathway_id)

        # ── Any unrecognised non-indented line ────────────────────────────────
        else:
            if not line.startswith(" ") and line.strip():
                current_section = None  # new unhandled section — reset tracker

    return result


# HELPER FUNCTION: parse_pathway_entry
# Parses a KEGG PATHWAY flat-file entry (e.g. from /get/hsa04930).
# Very similar structure to parse_disease_entry but pathway entries use a
# slightly different format for genes (symbol comes after a semicolon-separated
# description) and have a REL_PATHWAY section instead of PATHWAY.
# Returns a dictionary with the pathway name, genes, drugs, and related pathways.

def parse_pathway_entry(text):
    result = {"name": "", "genes": [], "drugs": [], "related_pathways": []}  # initialise result dict with empty values for each category
    current_section = None  # tracks which section we are currently reading

    for line in text.split("\n"):  # loop through each line of the flat-file text
        if line.startswith("NAME"):  # check if this line starts the NAME section
            result["name"] = line.replace("NAME", "").strip().rstrip(" - Homo sapiens (human)")  # remove the "NAME" label and strip the species suffix KEGG appends to pathway names
            current_section = "NAME"  # update section tracker
        elif line.startswith("GENE"):  # check if this line starts the GENE section
            current_section = "GENE"  # update section tracker to GENE
            content = line.replace("GENE", "").strip()  # remove the "GENE" label to get just the data content
            if content:  # only process if there is content on this line
                symbol = content.split(";")[0].split()[-1]  # pathway gene lines look like "1234  SYMBOL; full name" — split on semicolon first, then take the last token of the first part to get the symbol
                result["genes"].append(symbol)  # add the gene symbol to the genes list
        elif line.startswith("DRUG"):  # check if this line starts the DRUG section
            current_section = "DRUG"  # update section tracker to DRUG
            content = line.replace("DRUG", "").strip()  # remove the "DRUG" label to get just the data content
            if content:  # only process if there is content on this line
                drug_id = content.split()[0]  # the first token is the KEGG drug ID (e.g. D00219)
                result["drugs"].append(drug_id)  # add the drug ID to the drugs list
        elif line.startswith("REL_PATHWAY"):  # check if this line starts the REL_PATHWAY (related pathways) section
            current_section = "REL_PATHWAY"  # update section tracker to REL_PATHWAY
            content = line.replace("REL_PATHWAY", "").strip()  # remove the "REL_PATHWAY" label to get just the data content
            if content:  # only process if there is content on this line
                pid = content.split()[0]  # the first token is the related pathway ID
                result["related_pathways"].append(pid)  # add the pathway ID to the related pathways list
        elif line.startswith(" ") and current_section:  # lines starting with a space are continuations of the previous section
            content = line.strip()  # strip leading/trailing whitespace
            if not content:  # skip blank lines
                continue
            if current_section == "GENE":  # if still in the GENE section
                symbol = content.split(";")[0].split()[-1]  # same semicolon-split symbol parsing as above
                result["genes"].append(symbol)  # add the gene symbol to the list
            elif current_section == "DRUG":  # if still in the DRUG section
                drug_id = content.split()[0]  # extract the first token as the drug ID
                result["drugs"].append(drug_id)  # add the drug ID to the list
            elif current_section == "REL_PATHWAY":  # if still in the REL_PATHWAY section
                pid = content.split()[0]  # extract the first token as the related pathway ID
                result["related_pathways"].append(pid)  # add the pathway ID to the list
        else:
            if not line.startswith(" ") and line.strip():  # if a non-indented, non-empty line appears
                current_section = None  # it is a new unrecognised section — reset the tracker


    return result  # return the populated result dictionary



# HELPER FUNCTION: get_pathway_name
# Fetches a pathway entry from KEGG and returns just its name.
# Used when adding related pathway nodes so we can store a human-readable name.

def get_pathway_name(pathway_id):
    text = kegg_get(f"get/{pathway_id}")  # fetch the full pathway entry from KEGG
    for line in text.split("\n"):  # loop through each line of the response
        if line.startswith("NAME"):  # look for the NAME line
            return line.replace("NAME", "").strip().replace(" - Homo sapiens (human)", "")  # return the name after removing the label and the species suffix
    return pathway_id  # if no NAME line is found, fall back to returning the raw pathway ID



# HELPER FUNCTION: get_drug_name
# Fetches a drug entry from KEGG and returns its generic name.
# Drug IDs like D00944 are not human-readable, so this converts them to names
# like "Metformin hydrochloride" for the dataset.

def get_drug_name(drug_id):
    text = kegg_get(f"get/{drug_id}")  # fetch the full drug entry from KEGG using the drug ID
    for line in text.split("\n"):  # loop through each line of the response
        if line.startswith("NAME"):  # look for the NAME line
            return line.replace("NAME", "").strip().rstrip(";")  # return the name after removing the label and any trailing semicolons (KEGG sometimes appends one)
    return drug_id  # if no NAME line is found, fall back to returning the raw drug ID



# HELPER FUNCTION: get_gene_info
# Looks up a gene symbol in the KEGG human gene database and returns its
# KEGG gene ID (e.g. "hsa:5594") which can be used to construct the source_ref URL.

def get_gene_info(gene_symbol):
    text = kegg_get(f"find/hsa/{gene_symbol}")  # search the human (hsa) gene database for this symbol
    for line in text.split("\n"):  # loop through each result line
        if gene_symbol in line:  # find the line that contains our exact symbol
            return line.split("\t")[0].strip()  # return the first column (the KEGG gene ID, e.g. hsa:5594)
    return f"hsa:{gene_symbol}"  # if no match found, construct a best-guess ID using the symbol directly



In [35]:

# CONFIGURATION
# Define which disease and pathway IDs to extract.
# To add more diseases or pathways later, just add entries to these dictionaries.


DISEASE_IDS = {
    "T2DM":     "H00409",  # KEGG disease ID for Type 2 Diabetes Mellitus
    "Melanoma": "H00038",  # KEGG disease ID for Melanoma
}

PATHWAY_IDS = {
    "T2DM":     "hsa04930",  # KEGG pathway ID for the T2DM human pathway
    "Melanoma": "hsa05218",  # KEGG pathway ID for the Melanoma human pathway
}

nodes = []   # master list of node dictionaries — will become Sheet1 of the Excel file
edges = []   # master list of edge dictionaries — will become Sheet2 of the Excel file


# -----------------------------------------------------------------------------
# HELPER FUNCTION: add_node
# Appends a new node entry to the global nodes list.
# Using a helper function ensures every node has the same set of keys,
# which prevents KeyError issues when building the DataFrame later.
# -----------------------------------------------------------------------------
def add_node(node_id, node_type, name, source_db, source_ref, notes):
    nodes.append({          # append a new dictionary to the nodes list
        "node_id":    node_id,    # unique identifier for this node (e.g. "Gene:MAPK1")
        "node_type":  node_type,  # category of node: Disease, Pathway, Gene, or Drug
        "name":       name,       # human-readable display name
        "source_db":  source_db,  # database this entry came from (e.g. "KEGG" or "Manual")
        "source_ref": source_ref, # URL or accession number pointing to the original source record
        "notes":      notes,      # free-text notes about this node (e.g. which disease/pathway it belongs to)
    })


# -----------------------------------------------------------------------------
# HELPER FUNCTION: add_edge
# Appends a new edge entry to the global edges list.
# Edges represent relationships between nodes (e.g. a pathway HAS_GENE a gene).
# -----------------------------------------------------------------------------
def add_edge(source_id, relation, target_id, source_db, source_ref, notes=""):
    edges.append({            # append a new dictionary to the edges list
        "source_id":  source_id,   # node_id of the node where the relationship starts
        "relation":   relation,    # type of relationship (e.g. "HAS_GENE", "HAS_DRUG", "SAME_AS")
        "target_id":  target_id,   # node_id of the node where the relationship ends
        "source_db":  source_db,   # database that asserts this relationship
        "source_ref": source_ref,  # URL or accession pointing to evidence for this relationship
        "notes":      notes,       # optional free-text notes about this edge
    })


In [36]:

# STEP 1 — Create seed disease nodes
# These are the two "anchor" nodes for the entire graph, created manually
# rather than extracted from KEGG, since they represent the research question
# itself rather than a specific database entry.

for disease_label, disease_id in DISEASE_IDS.items():  # loop over each disease label and its KEGG ID
    seed_id  = f"Disease:{disease_label.replace(' ', '_')}"  # create the seed node ID (e.g. "Disease:T2DM")
    kegg_id  = f"KEGG:Disease:{disease_id}"                  # create the KEGG disease node ID (e.g. "KEGG:Disease:H00409")
    seed_url = f"https://www.kegg.jp/entry/{disease_id}"     # construct the KEGG web URL for this disease entry

    add_node(seed_id,  "Disease", disease_label, "Manual", "", "Seed disease node")  # add the manually-created seed node (no source URL since it's a manual anchor)
    add_node(kegg_id,  "Disease", disease_label, "KEGG", seed_url, f"KEGG Disease entry ({disease_id})")  # add the corresponding KEGG database node with its URL
    add_edge(seed_id, "SAME_AS", kegg_id, "KEGG", seed_url)  # add a SAME_AS edge linking the seed node to its KEGG counterpart


In [37]:
# STEP 2 — Extract data from KEGG DISEASE entries (H00409 and H00038)
# Each KEGG disease entry lists the genes associated with that disease
# (from genetic studies) and the drugs approved to treat it.
# This is different from the pathway — disease genes come from GWAS/linkage
# studies, not from the biochemical pathway diagram.


# PARSE_DISEASE_ENTRY — updated with parenthetical suffix fix
# Some KEGG disease PATHWAY lines attach a classification code directly to
# the pathway ID with no space, e.g. "hsa04519(N02001)  Adhesion molecules"
# content.split()[0] returns "hsa04519(N02001)" which breaks the KEGG URL.
# Fix: .split("(")[0] strips everything from the first parenthesis onwards.


def parse_disease_entry(text):
    """
    Parse a KEGG DISEASE flat-file entry (e.g. from /get/H00409 or /get/H00038).
    Returns a dict with keys: name, genes, drugs, pathways.
    """
    result = {
        "name":     "",   # disease display name
        "genes":    [],   # gene symbols associated with this disease
        "drugs":    [],   # KEGG drug IDs e.g. ["D00944", ...]
        "pathways": [],   # pathway IDs from the PATHWAY section e.g. ["hsa04110", ...]
    }
    current_section = None  # tracks which section header we are currently inside

    for line in text.split("\n"):  # loop through each line of the flat file

        # ── NAME ──────────────────────────────────────────────────────────────
        if line.startswith("NAME"):
            result["name"] = line.replace("NAME", "").strip()  # extract disease name
            current_section = "NAME"

        # ── GENE ──────────────────────────────────────────────────────────────
        elif line.startswith("GENE"):
            current_section = "GENE"
            content = line.replace("GENE", "").strip()  # remove section label
            if content:
                kind, value = extract_gene_symbol(content)  # classify the line content
                if kind == "gene":
                    result["genes"].append(value)           # plain gene symbol
                elif kind == "hsa_id":
                    symbol = get_gene_symbol_from_hsa_id(value)  # look up symbol via API
                    if symbol:
                        result["genes"].append(symbol)
                # pathway cross-references in GENE section are skipped here —
                # they are already captured by the PATHWAY section below

        # ── DRUG ──────────────────────────────────────────────────────────────
        elif line.startswith("DRUG"):
            current_section = "DRUG"
            content = line.replace("DRUG", "").strip()
            dr_match = re.search(r'\[DR:(D\d+)\]', content)  # drug ID is inside [DR:D10025] tag
            if dr_match:
                result["drugs"].append(dr_match.group(1))    # extract just the D-number

        # ── PATHWAY ───────────────────────────────────────────────────────────
        elif line.startswith("PATHWAY"):
            current_section = "PATHWAY"
            content = line.replace("PATHWAY", "").strip()
            if content:
                # FIX: strip parenthetical suffix attached directly to the ID
                # e.g. "hsa04519(N02001)" → "hsa04519"
                # Without this, the raw token breaks the KEGG REST URL with a 400 error
                pathway_id = content.split()[0].split("(")[0]
                if pathway_id not in result["pathways"]:     # avoid duplicates
                    result["pathways"].append(pathway_id)

        # ── Continuation lines (indented with spaces) ─────────────────────────
        elif line.startswith(" ") and current_section:
            content = line.strip()
            if not content:                                  # skip blank continuation lines
                continue

            if current_section == "GENE":
                kind, value = extract_gene_symbol(content)
                if kind == "gene":
                    result["genes"].append(value)
                elif kind == "hsa_id":
                    symbol = get_gene_symbol_from_hsa_id(value)
                    if symbol:
                        result["genes"].append(symbol)

            elif current_section == "DRUG":
                dr_match = re.search(r'\[DR:(D\d+)\]', content)
                if dr_match:
                    result["drugs"].append(dr_match.group(1))

            elif current_section == "PATHWAY":
                # FIX: same parenthetical strip for continuation lines
                pathway_id = content.split()[0].split("(")[0]
                if pathway_id not in result["pathways"]:
                    result["pathways"].append(pathway_id)

        # ── Unrecognised non-indented line ────────────────────────────────────
        else:
            if not line.startswith(" ") and line.strip():
                current_section = None  # new unhandled section — reset tracker

    return result



# STEP 2 — KEGG DISEASE ENTRIES


print("Fetching KEGG DISEASE entries...")

for disease_label, disease_id in DISEASE_IDS.items():
    kegg_disease_id = f"KEGG:Disease:{disease_id}"               # node ID for this KEGG disease entry
    disease_url     = f"https://www.kegg.jp/entry/{disease_id}"  # source URL for this disease

    entry = parse_disease_entry(kegg_get(f"get/{disease_id}"))   # fetch and parse the disease flat file

    # Related pathways from the PATHWAY section of the disease flat file
    for pid in entry["pathways"]:                                 # pid is already in hsa format e.g. hsa04110
        rel_node_id = f"KEGG:Pathway:{pid}"                      # build the pathway node ID
        rel_name    = get_pathway_name(pid)                       # fetch the human-readable name from KEGG
        rel_url     = f"https://www.kegg.jp/entry/{pid}"         # build the pathway entry URL
        add_node(                                                 # add the pathway as a node in the dataset
            rel_node_id, "Pathway", rel_name,
            "KEGG", disease_url,
            f"Related pathway from {disease_label} disease ({disease_id})"
        )
        add_edge(                                                 # connect the disease to this pathway
            kegg_disease_id, "RELATED_PATHWAY", rel_node_id,
            "KEGG", disease_url
        )

    # Gene nodes and edges
    for symbol in entry["genes"]:
        gene_node_id = f"Gene:{symbol}"                          # build the gene node ID
        add_node(gene_node_id, "Gene", symbol, "KEGG", disease_url,
                 f"{disease_label} disease ({disease_id})")
        add_edge(kegg_disease_id, "HAS_GENE", gene_node_id, "KEGG", disease_url)

    # Drug nodes and edges
    for drug_id in entry["drugs"]:
        drug_node_id = f"KEGG:Drug:{drug_id}"                    # build the drug node ID
        drug_name    = get_drug_name(drug_id)                    # look up human-readable name
        drug_url     = f"https://www.kegg.jp/entry/{drug_id}"   # build the drug entry URL
        add_node(drug_node_id, "Drug", drug_name, "KEGG", drug_url,
                 f"Drug listed in {disease_label} disease ({disease_id})")
        add_edge(kegg_disease_id, "HAS_DRUG", drug_node_id, "KEGG", disease_url)

    print(f"  {disease_id}: {len(entry['genes'])} genes, "
          f"{len(entry['drugs'])} drugs, "
          f"{len(entry['pathways'])} related pathways")

Fetching KEGG DISEASE entries...
  H00409: 21 genes, 66 drugs, 10 related pathways
  H00038: 12 genes, 18 drugs, 0 related pathways


In [38]:

# STEP 3 — Extract data from KEGG PATHWAY entries (hsa04930 and hsa05218)
# Each KEGG pathway entry lists the genes drawn on the pathway diagram and
# the drugs that act on those genes. It also lists related pathways.
# These are the mechanistic pathways, distinct from the disease gene lists.

print("\nFetching KEGG PATHWAY entries...")  # print progress message

for disease_label, pathway_id in PATHWAY_IDS.items():  # loop over each disease and its associated main pathway ID
    pathway_node_id = f"KEGG:Pathway:{pathway_id}"              # build the pathway node ID (e.g. "KEGG:Pathway:hsa04930")
    pathway_url     = f"https://www.kegg.jp/entry/{pathway_id}" # build the source URL for this pathway entry

    entry = parse_pathway_entry(kegg_get(f"get/{pathway_id}"))  # fetch the pathway entry from KEGG and parse it into a structured dictionary

    add_node(pathway_node_id, "Pathway", entry["name"], "KEGG", pathway_url,
             f"{disease_label} main pathway")  # add the pathway as a node using its parsed name

    seed_id = f"Disease:{disease_label.replace(' ', '_')}"  # get the seed disease node ID to link from
    add_edge(seed_id, "HAS_KEGG_PATHWAY", pathway_node_id, "KEGG", pathway_url)  # add a HAS_KEGG_PATHWAY edge from the seed disease node to this pathway

    # Add related pathway nodes and edges
    for rel_pid in entry["related_pathways"]:  # loop over each related pathway ID found in the entry
        human_pid   = rel_pid.replace("map", "hsa") if rel_pid.startswith("map") else rel_pid  # convert "map" prefix to "hsa" for human pathway IDs
        rel_node_id = f"KEGG:Pathway:{human_pid}"            # build the related pathway node ID
        rel_name    = get_pathway_name(human_pid)            # fetch the human-readable name for this related pathway
        rel_url     = f"https://www.kegg.jp/entry/{human_pid}"  # build the source URL for the related pathway entry
        add_node(rel_node_id, "Pathway", rel_name, "KEGG", pathway_url,
                 f"Related pathway from {pathway_id} ({disease_label})")  # add the related pathway as a node, noting which main pathway it came from
        add_edge(pathway_node_id, "RELATED_PATHWAY", rel_node_id, "KEGG", pathway_url)  # add a RELATED_PATHWAY edge from the main pathway to this related pathway

    # Add gene nodes and edges from the pathway entry
    for symbol in entry["genes"]:  # loop over each gene symbol found in the pathway entry
        gene_node_id = f"Gene:{symbol}"                          # build the gene node ID
        gene_url     = f"https://www.kegg.jp/entry/hsa:{symbol}"  # construct a gene entry URL using the symbol (approximate — exact NCBI ID would require an extra lookup)
        add_node(gene_node_id, "Gene", symbol, "KEGG", gene_url,
                 f"{disease_label} pathway ({pathway_id})")  # add the gene as a node with the pathway as context
        add_edge(pathway_node_id, "HAS_GENE", gene_node_id, "KEGG", pathway_url)  # add a HAS_GENE edge from the pathway to this gene

    # Add drug nodes and edges from the pathway entry
    for drug_id in entry["drugs"]:  # loop over each drug ID found in the pathway entry
        drug_node_id = f"KEGG:Drug:{drug_id}"                    # build the drug node ID
        drug_name    = get_drug_name(drug_id)                    # look up the human-readable drug name from KEGG
        drug_url     = f"https://www.kegg.jp/entry/{drug_id}"   # build the source URL for this drug entry
        add_node(drug_node_id, "Drug", drug_name, "KEGG", drug_url,
                 f"Drug listed in {disease_label} pathway ({pathway_id})")  # add the drug as a node with the pathway as context
        add_edge(pathway_node_id, "HAS_DRUG", drug_node_id, "KEGG", pathway_url)  # add a HAS_DRUG edge from the pathway to this drug

    print(f"  {pathway_id}: {len(entry['genes'])} genes, {len(entry['drugs'])} drugs, "
          f"{len(entry['related_pathways'])} related pathways")  # print a summary for this pathway


Fetching KEGG PATHWAY entries...
  hsa04930: 47 genes, 35 drugs, 4 related pathways
  hsa05218: 73 genes, 6 drugs, 6 related pathways


In [40]:
# STEP 4 — DRUG TARGET EXTRACTION
# For each drug node already in the dataset, fetch its target genes.
# Strategy:
#   1. Try KEGG DRUG entry first — parses the TARGET section of the flat file
#   2. Fall back to ChEMBL API for drugs KEGG doesn't annotate with targets


# Tokens that appear as the first token in KEGG TARGET section lines but are
# NOT gene symbols. These are KEGG sub-labels, molecular target types, or
# non-human gene references that must be filtered out.
NON_GENE_TOKENS = {
    'PATHWAY',  # KEGG sub-label for pathway-level targets e.g. "PATHWAY hsa04910 Insulin signaling"
    'DNA',      # molecular target for alkylating agents e.g. Dacarbazine
    'RNA',      # molecular target for some chemotherapy drugs
    'NETWORK',  # KEGG network label — not a gene
    'VECTOR',   # viral vector — appears in gene therapy entries
    'GENOME',   # whole-genome reference — not a gene
}


def _try_add_target(content, targets):
    """
    Attempt to extract a valid gene symbol from a single TARGET section line.
    Applies three layers of filtering before accepting a token as a gene symbol:
      Layer 1 — skip if [PATH:] bracket present — explicit pathway reference
      Layer 2 — skip if first token is in NON_GENE_TOKENS blocklist
      Layer 3 — validate first token matches gene symbol pattern
    Modifies the targets list in place, returns nothing.
    """
    if '[PATH:' in content:              # Layer 1: explicit pathway reference line — skip
        return

    parts = content.split()             # split line into whitespace-separated tokens
    if not parts:                       # skip empty lines
        return

    first_token = parts[0]              # first token is the candidate gene symbol

    if first_token in NON_GENE_TOKENS:  # Layer 2: known non-gene token — skip
        return

    # Layer 3: validate the token looks like a real gene symbol
    # Gene symbols start with an uppercase letter and contain only
    # uppercase letters, digits, and hyphens e.g. ABCC8, GLP1R, P3R3URF-PIK3R3
    if re.match(r'^[A-Z][A-Z0-9\-]+$', first_token):
        if first_token not in targets:  # avoid duplicates within the same drug
            targets.append(first_token)


def get_kegg_drug_targets(drug_id):
    """
    Fetch a KEGG drug entry and extract target gene symbols from the TARGET section.

    KEGG TARGET section formats:
      Real gene target:   "ABCC8 [HSA:6833] [KO:K05004]"
      Pathway target:     "PATHWAY  hsa04910  Insulin signaling pathway"
      Molecular target:   "DNA  [GEN]"

    Only the first format represents actual protein-coding gene targets.
    Returns a list of validated gene symbol strings, empty list if none found.
    """
    try:
        text = kegg_get(f"get/{drug_id}")   # fetch the full KEGG drug flat-file entry
    except Exception as e:
        print(f"    Warning: could not fetch {drug_id} from KEGG — {e}")
        return []                           # return empty list on any fetch error

    targets = []                            # list to accumulate valid gene symbols
    current_section = None                  # tracks which section we are reading

    for line in text.split("\n"):           # loop through each line of the drug entry

        if line.startswith("TARGET"):       # detect the TARGET section header
            current_section = "TARGET"
            content = line.replace("TARGET", "").strip()  # remove label, get data portion
            if content:
                _try_add_target(content, targets)         # attempt to extract a gene symbol

        elif line.startswith(" ") and current_section == "TARGET":  # continuation line within TARGET
            content = line.strip()
            if content:
                _try_add_target(content, targets)         # attempt to extract a gene symbol

        elif not line.startswith(" ") and line.strip():   # non-indented line means we left TARGET
            current_section = None

    return targets                          # return the list of validated gene symbols


def get_chembl_targets(drug_name):
    """
    Query the ChEMBL REST API for human protein targets of a drug by preferred name.
    Used as a fallback when KEGG does not annotate drug targets.
    Returns a list of gene symbols (strings), empty list if none found or any error.
    """
    # Step 1 — search ChEMBL for the molecule by name to get its ChEMBL ID
    search_url = (
        f"https://www.ebi.ac.uk/chembl/api/data/molecule"
        f"?pref_name__iexact={drug_name}&format=json"
    )
    try:
        resp = requests.get(search_url, timeout=10)   # send search request with timeout
        resp.raise_for_status()
        molecules = resp.json().get("molecules", [])  # extract list of matching molecules
        if not molecules:
            return []                                 # no molecule found under this name
        chembl_id = molecules[0]["molecule_chembl_id"]  # take the first match's ChEMBL ID
    except Exception:
        return []                                     # any network or parsing error — return empty

    time.sleep(0.3)                                   # rate limit pause between requests

    # Step 2 — fetch activity data filtered to human targets and quantitative activity types
    activity_url = (
        f"https://www.ebi.ac.uk/chembl/api/data/activity"
        f"?molecule_chembl_id={chembl_id}"
        f"&target_organism=Homo+sapiens"
        f"&standard_type__in=IC50,Ki,Kd,EC50"
        f"&format=json&limit=100"
    )
    try:
        resp = requests.get(activity_url, timeout=10)
        resp.raise_for_status()
        activities = resp.json().get("activities", [])  # list of activity records
    except Exception:
        return []

    time.sleep(0.3)                                   # rate limit pause

    # Step 3 — for each activity record, fetch the target entry to get gene symbols
    gene_symbols = set()                              # set to automatically deduplicate
    seen_targets = set()                              # track ChEMBL target IDs already looked up

    for act in activities:
        target_chembl_id = act.get("target_chembl_id")  # get the target ID from the activity record
        if not target_chembl_id or target_chembl_id in seen_targets:
            continue                                  # skip missing or already-processed targets
        seen_targets.add(target_chembl_id)            # mark this target as seen

        target_url = (
            f"https://www.ebi.ac.uk/chembl/api/data/target"
            f"/{target_chembl_id}?format=json"
        )
        try:
            tresp = requests.get(target_url, timeout=10)
            tresp.raise_for_status()
            target_data = tresp.json()
            if target_data.get("target_type") != "SINGLE PROTEIN":  # only single-protein targets
                continue
            for comp in target_data.get("target_components", []):   # loop through protein components
                for syn in comp.get("target_component_synonyms", []):  # loop through synonyms
                    if syn.get("syn_type") == "GENE_SYMBOL":         # only official gene symbols
                        gene_symbols.add(syn["component_synonym"].upper())  # normalise to uppercase
        except Exception:
            continue
        time.sleep(0.2)                               # small pause per target lookup

    return list(gene_symbols)                         # return deduplicated symbol list



# MAIN TARGET EXTRACTION LOOP


print("Step 4: Extracting drug targets...")

# Safety check — nodes and edges must be plain lists of dicts at this point.
# If they are DataFrames it means the export cell ran too early.
if isinstance(nodes, pd.DataFrame):
    print("  WARNING: nodes was a DataFrame — converting back to list of dicts")
    nodes = nodes.to_dict(orient="records")

if isinstance(edges, pd.DataFrame):
    print("  WARNING: edges was a DataFrame — converting back to list of dicts")
    edges = edges.to_dict(orient="records")

if not nodes or not isinstance(nodes[0], dict):
    print("ERROR: nodes is empty or not a list of dicts — re-run from Step 1")

else:
    # Collect all drug node IDs from the current nodes list
    drug_node_ids = [
        n["node_id"] for n in nodes
        if n["node_type"] == "Drug"
    ]

    # Build a lookup from node_id to drug name for the ChEMBL fallback
    drug_name_lookup = {
        n["node_id"]: n["name"]
        for n in nodes
        if n["node_type"] == "Drug"
    }

    print(f"  Found {len(drug_node_ids)} drug nodes to process...")

    for drug_node_id in drug_node_ids:                          # loop over every drug node

        drug_id   = drug_node_id.replace("KEGG:Drug:", "")     # strip prefix to get raw ID e.g. D00944
        drug_name = drug_name_lookup.get(drug_node_id, "")     # human-readable name for ChEMBL fallback

        # Try KEGG first — primary source since IDs are already KEGG D-numbers
        kegg_targets = get_kegg_drug_targets(drug_id)

        if kegg_targets:                                        # KEGG returned at least one target
            target_symbols = kegg_targets
            target_source  = "KEGG"
            target_ref     = f"https://www.kegg.jp/entry/{drug_id}"
        else:                                                   # KEGG had no targets — try ChEMBL
            print(f"    No KEGG targets for {drug_id} ({drug_name}) — trying ChEMBL...")
            target_symbols = get_chembl_targets(drug_name)
            target_source  = "ChEMBL"
            target_ref     = ""                                 # no single URL for a ChEMBL query

        for symbol in target_symbols:                           # loop over each target gene found
            gene_node_id = f"Gene:{symbol}"                    # build the gene node ID

            add_node(                                           # add gene node — deduplication merges if already present
                gene_node_id, "Gene", symbol,
                target_source, target_ref,
                f"Drug target — sourced from {target_source} for {drug_id}"
            )

            add_edge(                                           # add TARGETS edge from drug to gene
                drug_node_id, "TARGETS", gene_node_id,
                target_source, target_ref,
                f"Drug-target relationship from {target_source}"
            )

        if target_symbols:
            print(f"    {drug_id}: {len(target_symbols)} targets ({target_source})")
        else:
            print(f"    {drug_id}: no targets found in KEGG or ChEMBL")

    print(f"\n  Running totals after Step 4: {len(nodes)} nodes, {len(edges)} edges")

Step 4: Extracting drug targets...
  Found 125 drug nodes to process...
    D03230: 1 targets (KEGG)
    D04477: 1 targets (KEGG)
    D04475: 1 targets (KEGG)
    D04540: 1 targets (KEGG)
    D03250: 1 targets (KEGG)
    D04539: 1 targets (KEGG)
    D09727: 1 targets (KEGG)
    D11034: 2 targets (KEGG)
    D11567: 2 targets (KEGG)
    D00944: 1 targets (KEGG)
    No KEGG targets for D00336 (Glyburide (USP)) — trying ChEMBL...
    D00336: no targets found in KEGG or ChEMBL
    No KEGG targets for D00271 (Chlorpropamide (JP18/USP/INN)) — trying ChEMBL...
    D00271: no targets found in KEGG or ChEMBL
    No KEGG targets for D00380 (Tolbutamide (JAN/USP/INN)) — trying ChEMBL...
    D00380: no targets found in KEGG or ChEMBL
    No KEGG targets for D00379 (Tolazamide (JAN/USP/INN)) — trying ChEMBL...
    D00379: no targets found in KEGG or ChEMBL
    D00335: 1 targets (KEGG)
    D00593: 1 targets (KEGG)
    No KEGG targets for D10265 (Glipizide and metformin hydrochloride) — trying ChEMBL.

In [41]:

# STEP 5 — Deduplicate nodes and edges
# Because the same gene (e.g. ABCC8) or drug may appear in both a disease entry
# and a pathway entry, we end up with duplicate entries in our raw lists.
# These functions merge duplicates into single rows and combine their notes.


def deduplicate_nodes(node_list):
    seen = {}  # dictionary mapping node_id to the first seen node dict — used to detect duplicates
    for node in node_list:  # loop over every node we collected
        nid = node["node_id"]  # get this node's ID
        if nid not in seen:  # if we haven't seen this ID before
            seen[nid] = node.copy()  # store a copy of this node as the canonical entry
        else:
            existing_notes = seen[nid]["notes"]  # get the notes already stored for this node
            new_notes      = node["notes"]        # get the notes from this duplicate occurrence
            if new_notes and new_notes not in existing_notes:  # only append if the new notes add information not already present
                seen[nid]["notes"] = f"{existing_notes}; {new_notes}"  # merge notes by joining with a semicolon separator
    return list(seen.values())  # return the deduplicated list (one entry per unique node_id)


def deduplicate_edges(edge_list):
    seen   = set()   # set of (source_id, relation, target_id) tuples we have already added
    unique = []      # list to hold the deduplicated edges
    for edge in edge_list:  # loop over every edge we collected
        key = (edge["source_id"], edge["relation"], edge["target_id"])  # create a tuple that uniquely identifies this relationship
        if key not in seen:  # if this exact relationship hasn't been added yet
            seen.add(key)    # record that we've now seen this relationship
            unique.append(edge)  # add the edge to the unique list
    return unique  # return the deduplicated edge list


nodes = deduplicate_nodes(nodes)   # apply deduplication to the full nodes list
edges = deduplicate_edges(edges)   # apply deduplication to the full edges list


In [42]:
# STEP 6 — Flag shared genes
# Find genes that appear in HAS_GENE edges from BOTH the T2DM pathway and the
# Melanoma pathway — these are the biologically interesting bridge nodes.
# Update their notes to highlight this shared status.


t2dm_genes = {e["target_id"] for e in edges
              if e["source_id"] == "KEGG:Pathway:hsa04930" and e["relation"] == "HAS_GENE"}  # collect all gene node IDs that are connected to the T2DM pathway via HAS_GENE edges

mel_genes  = {e["target_id"] for e in edges
              if e["source_id"] == "KEGG:Pathway:hsa05218"  and e["relation"] == "HAS_GENE"}  # collect all gene node IDs that are connected to the Melanoma pathway via HAS_GENE edges

shared = t2dm_genes & mel_genes  # find the intersection — genes present in both sets are the shared genes

node_dict = {n["node_id"]: n for n in nodes}  # build a lookup dictionary from node_id to node dict for fast access

for gid in shared:  # loop over each shared gene ID
    if gid in node_dict:  # check the gene actually exists in our node list (it should, but defensive check)
        node_dict[gid]["notes"] = "Shared — T2DM pathway (hsa04930) + Melanoma pathway (hsa05218)"  # overwrite the notes to clearly label this gene as shared between both pathways

nodes = list(node_dict.values())  # convert the dictionary back to a list for export

print(f"\nShared pathway genes: {len(shared)}")          # print how many shared genes were found
print(f"  → {', '.join(sorted(shared))}")               # print their names in alphabetical order



Shared pathway genes: 9
  → Gene:MAPK1, Gene:MAPK3, Gene:P3R3URF-PIK3R3, Gene:PIK3CA, Gene:PIK3CB, Gene:PIK3CD, Gene:PIK3R1, Gene:PIK3R2, Gene:PIK3R3


In [43]:
# Adding STRING PPI to the graph 
# using a 0.7 confidence threshold

import requests

def get_string_interactions(gene_list, species=9606, score_threshold=700):
    """
    Fetch protein-protein interactions from STRING for a list of gene symbols.
    species=9606 is Homo sapiens.
    score_threshold: 0-1000. 400=medium, 700=high, 900=very high confidence.
    Returns a list of (gene1, gene2, score) tuples.
    """
    url = "https://string-db.org/api/json/network"
    params = {
        "identifiers": "%0d".join(gene_list),  # genes separated by URL-encoded newline
        "species":     species,
        "caller_identity": "kg_project",        # STRING asks you to identify your app
        "required_score": score_threshold,
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    interactions = []
    for entry in data:
        gene_a = entry["preferredName_A"]   # symbol of first interacting protein
        gene_b = entry["preferredName_B"]   # symbol of second interacting protein
        score  = entry["score"]             # combined interaction confidence 0–1
        interactions.append((gene_a, gene_b, score))
    return interactions


# Collect all gene symbols currently in the dataset
gene_symbols = [
    n["name"] for n in nodes
    if n["node_type"] == "Gene"
]

print(f"Fetching STRING interactions for {len(gene_symbols)} genes...")
interactions = get_string_interactions(gene_symbols, score_threshold=700)

for gene_a, gene_b, score in interactions:
    node_a = f"Gene:{gene_a}"
    node_b = f"Gene:{gene_b}"
    # Only add edge if both genes already exist in the dataset
    existing_ids = {n["node_id"] for n in nodes}
    if node_a in existing_ids and node_b in existing_ids:
        add_edge(
            source_id  = node_a,
            relation   = "INTERACTS_WITH",
            target_id  = node_b,
            source_db  = "STRING",
            source_ref = f"https://string-db.org/network/{gene_a}%0d{gene_b}",
            notes      = f"STRING combined score: {score:.3f}"
        )

print(f"Added {len(interactions)} PPI edges from STRING")

Fetching STRING interactions for 157 genes...
Added 1204 PPI edges from STRING


In [45]:
# STEP 7 — METABOLITE EXTRACTION (main pathways only)
# Only extracts compounds from hsa04930 (T2DM) and hsa05218 (Melanoma).

print("Step 7: Extracting metabolites from main pathways only...")

for disease_label, pathway_id in PATHWAY_IDS.items():     # loops over only hsa04930 and hsa05218

    pathway_node_id = f"KEGG:Pathway:{pathway_id}"
    pathway_url     = f"https://www.kegg.jp/entry/{pathway_id}"

    text = kegg_get_safe(f"get/{pathway_id}")              # fetch the pathway flat file
    if not text:
        print(f"  Skipping {pathway_id} — could not fetch")
        continue

    compounds = parse_compound_section(text)               # parse only the COMPOUND section

    for cpd_id in compounds:
        cpd_node_id = f"KEGG:Compound:{cpd_id}"
        cpd_url     = f"https://www.kegg.jp/entry/{cpd_id}"
        cpd_name    = get_compound_name(cpd_id)

        add_node(cpd_node_id, "Metabolite", cpd_name,
                 "KEGG", cpd_url,
                 f"Metabolite in {disease_label} pathway ({pathway_id})")
        add_edge(pathway_node_id, "HAS_METABOLITE", cpd_node_id,
                 "KEGG", pathway_url)

    print(f"  {pathway_id} ({disease_label}): {len(compounds)} compounds")


print(f"\n  Running totals after Step 7: {len(nodes)} nodes, {len(edges)} edges")

Step 7: Extracting metabolites from main pathways only...
  hsa04930 (T2DM): 6 compounds
  hsa05218 (Melanoma): 1 compounds

  Running totals after Step 7: 290 nodes, 1567 edges


In [47]:
# FINAL DEDUPLICATION
# Collapses duplicate nodes and edges introduced by metabolites appearing
# in multiple pathways, and any other overlaps from enrichment steps.

print("Final deduplication...")

nodes = deduplicate_nodes(nodes)   # merge duplicate node entries, combining their notes
edges = deduplicate_edges(edges)   # remove duplicate (source, relation, target) triples

print(f"  Final totals: {len(nodes)} nodes, {len(edges)} edges")

Final deduplication...
  Final totals: 290 nodes, 1567 edges


In [48]:
# STEP 8 — Export to Excel
# Write nodes to Sheet1 and edges to Sheet2.
# pd.ExcelWriter with the "with" statement ensures the file is properly saved
# and closed even if an error occurs.

with pd.ExcelWriter("T2DM_Melanoma_KG_Dataset_auto5.xlsx", engine="openpyxl") as writer:  # open an Excel writer that will save to this filename when the block exits
    pd.DataFrame(nodes).to_excel(writer, sheet_name="Sheet1", index=False)  # convert the nodes list to a DataFrame and write it to Sheet1, without the row number index
    pd.DataFrame(edges).to_excel(writer, sheet_name="Sheet2", index=False)  # convert the edges list to a DataFrame and write it to Sheet2, without the row number index

print(f"\nDone. {len(nodes)} nodes, {len(edges)} edges written to Excel.")  # print a final confirmation showing how many nodes and edges were saved



Done. 290 nodes, 1567 edges written to Excel.
